In [1]:
from pyspark.sql.session import SparkSession

spark = SparkSession.builder.master("local").appName('writing_data').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/14 17:57:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/14 17:57:38 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [18]:
df = spark.createDataFrame(data = [(1, "Maheer", "male", 2000),
                                    (2, "Wafa", "male", 3000)], 
                          schema = "id int, name string, gender string, salary int")

# mode, format, location -- 3 things to provide while writing dataframe into a file
df.write.mode("overwrite").format("json").save("/Users/shishir/Pyspark/SampleData/employee.json")

df.show()
df.printSchema()

spark.read.json("/Users/shishir/Pyspark/SampleData/employee.json").sort("salary").show()

+---+------+------+------+
| id|  name|gender|salary|
+---+------+------+------+
|  1|Maheer|  male|  2000|
|  2|  Wafa|  male|  3000|
+---+------+------+------+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)

+------+---+------+------+
|gender| id|  name|salary|
+------+---+------+------+
|  male|  1|Maheer|  2000|
|  male|  2|  Wafa|  3000|
+------+---+------+------+



In [37]:
# reading multline json file -- use parameter multiline=True.
# gives corrupt record error if not supplied this parameter while reading multiline files
df = spark.read.json("/Users/shishir/Pyspark/SampleData/multiline_emp.json", multiLine=True)
df.printSchema()
df.show()

root
 |-- contactInfo: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- hireDate: string (nullable = true)
 |-- id: long (nullable = true)
 |-- name: struct (nullable = true)
 |    |-- firstName: string (nullable = true)
 |    |-- lastName: string (nullable = true)
 |-- position: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- skills: array (nullable = true)
 |    |-- element: string (containsNull = true)

+--------------------+-----------+---+--------------+-----------------+------+--------------------+
|         contactInfo|   hireDate| id|          name|         position|salary|              skills|
+--------------------+-----------+---+--------------+-----------------+------+--------------------+
|{john.doe@example...|21-SEP-2005|101|   {John, Doe}|Software Engineer| 75000|[JavaScript, Pyth...|
|{jane.smith@examp...|15-JAN-2010|102| {Jane, Smith}|  Project Manager| 90000|[Agile, Scrum, Le...|
|{peter

In [53]:
# reading multiple multiline json files 
df = spark.read.json(["/Users/shishir/Pyspark/SampleData/JsonData/emp1.json",
                "/Users/shishir/Pyspark/SampleData/JsonData/emp2.json"], multiLine = True)

df.show()
df.printSchema()

+--------------------+---+-----------+--------------------+----------+---------+----------+-----------+--------+--------------+-----------------+-------+--------------------+
|             address|age| department|               email|employeeId|firstName|isFullTime|joiningDate|lastName|         phone|             role| salary|              skills|
+--------------------+---+-----------+--------------------+----------+---------+----------+-----------+--------+--------------+-----------------+-------+--------------------+
|{Bengaluru, 56000...| 28|Engineering|rahul.sharma@exam...|     E1001|    Rahul|      true| 2022-06-15|  Sharma|+91-9876543210|Software Engineer|1200000|[Python, Spark, SQL]|
|{Kolkata, 700016,...| 32|       Data|anita.verma@examp...|     E1002|    Anita|      true| 2021-03-10|   Verma|+91-9123456780|     Data Analyst| 950000|[Excel, Power BI,...|
+--------------------+---+-----------+--------------------+----------+---------+----------+-----------+--------+-------------

In [56]:
spark.range(5).write.mode("append").format('csv').save("/Users/shishir/Pyspark/SampleData/JsonData/")

In [60]:
# read all the json files in JsonData directory. Ofcourse, they all should have the same schema

# "/Users/shishir/Pyspark/SampleData/JsonData/*.json" specifically reads only the json files 
# "/Users/shishir/Pyspark/SampleData/JsonData/" reads all the files. 

# If you have different types of files in the target location and you want to read only json files, choose *.json
# This reads all the json files

df = spark.read.json("/Users/shishir/Pyspark/SampleData/JsonData/*.json", multiLine = True)

df.show()
df.printSchema()

+--------------------+---+-----------+--------------------+----------+---------+----------+-----------+--------+--------------+-----------------+-------+--------------------+
|             address|age| department|               email|employeeId|firstName|isFullTime|joiningDate|lastName|         phone|             role| salary|              skills|
+--------------------+---+-----------+--------------------+----------+---------+----------+-----------+--------+--------------+-----------------+-------+--------------------+
|{Bengaluru, 56000...| 28|Engineering|rahul.sharma@exam...|     E1001|    Rahul|      true| 2022-06-15|  Sharma|+91-9876543210|Software Engineer|1200000|[Python, Spark, SQL]|
|{Kolkata, 700016,...| 32|       Data|anita.verma@examp...|     E1002|    Anita|      true| 2021-03-10|   Verma|+91-9123456780|     Data Analyst| 950000|[Excel, Power BI,...|
+--------------------+---+-----------+--------------------+----------+---------+----------+-----------+--------+-------------

In [69]:
df = spark.createDataFrame([("Shishir",25,"Male","100k"),
                            ("Rahul",25,"Male","120k")], 
                       schema = "name string, age int, gender string, salary string")
df.show()

# pyspark stores the data in part files at the below dummy location. The data is typically split across multiple files"
# dummy folder will have all the files corresponding to each partition of the dataframe
# WHen you read this folder, data from all the files is read and returned.
# You can also save the data in one single file. Repartition the df to reduce the num of partitions to 1 and then write the data in json format.
# default mode is set as error. So if the file already exists pyspark will throw an exception. Hence its necessary to set modes
df.write.json("/Users/shishir/Pyspark/SampleData/JsonData/dummy/", mode = 'ignore')


+-------+---+------+------+
|   name|age|gender|salary|
+-------+---+------+------+
|Shishir| 25|  Male|  100k|
|  Rahul| 25|  Male|  120k|
+-------+---+------+------+



In [70]:
# appended the data at the same location. Hence getting same records twice
spark.read.json("/Users/shishir/Pyspark/SampleData/JsonData/dummy/*json").show()

+---+------+-------+------+
|age|gender|   name|salary|
+---+------+-------+------+
| 25|  Male|Shishir|  100k|
| 25|  Male|  Rahul|  120k|
| 25|  Male|Shishir|  100k|
| 25|  Male|  Rahul|  120k|
+---+------+-------+------+

